# Qualidade e preparação dos dados

Esse notebook tem como finalidade o diagnóstico da base de dados de pedidos antes da análise explorátória de dados (EDA). Portanto, será investigado a estrutura, regras de negócio e coerência entre variáveis da base de dados.

*Observações*
- Cada decisão realizada será registrada no [registro de decisões](../reports/registro_decisoes.md)
- A base de dados tratarada será gerada no final do notebook e salva em [../data/processed](../data/processed/)

In [ ]:
# Importações
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from loguru import logger

# Logs do projeto desligados para não gravar caminhos da máquina local nas saídas
logger.disable("src")

from src.dataset import download_raw_data

# Base bruta: baixada da fonte oficial na primeira execução e conferida pelo SHA-256 (D-01)
raw = pd.read_csv(download_raw_data())


## 1. Visão geral da base de dados

Nesta etapa, serão avaliadas a estrutura da base, os tipos de dados e as principais características estatísticas das variáveis, com foco na compreensão inicial dos dados e na identificação de possíveis inconsistências.

- **Dimensões:** quantidade de linhas e colunas.
- **Tipos de dados:** tipos de cada variável e adequação ao seu conteúdo.
- **Distribuição das notas:** escala, precisão decimal e estatísticas descritivas.
- **Observações iniciais:** padrões, valores atípicos e características relevantes identificadas.

In [ ]:
print(f"{raw.shape[0]} pedidos e {raw.shape[1]} colunas")
raw.head()


2500 pedidos e 19 colunas


,customer_id,customer_age,customer_region,customer_tenure_months,order_id,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,nps_score,repeat_purchase_30d,complaints_count,csat_internal_score
0,1,63,Nordeste,14,50001,139.73,4,39.35,4,2,2,55.53,3,0,4,6.9,0,3,6.5
1,2,20,Sul,1,50002,458.95,2,9.51,10,6,4,28.23,3,0,10,2.4,0,3,0.0
2,3,46,Nordeste,111,50003,507.06,5,42.82,6,6,1,40.99,1,4,5,4.8,0,7,1.5
3,4,52,Centro-Oeste,117,50004,302.19,2,19.58,9,5,2,35.24,3,1,11,5.9,0,4,0.3
4,5,56,Norte,50,50005,253.06,1,29.37,11,13,1,39.32,1,1,0,6.1,0,3,7.9


In [ ]:
# Perfil de cada coluna: tipo, valores preenchidos, valores distintos e faixa
perfil = pd.DataFrame({
    "tipo": raw.dtypes.astype(str),
    "preenchidos": raw.notna().sum(),
    "distintos": raw.nunique(),
    "mínimo": raw.min(numeric_only=True),
    "máximo": raw.max(numeric_only=True),
}).reindex(raw.columns)
perfil


,tipo,preenchidos,distintos,mínimo,máximo
customer_id,int64,2500,2500,1.00,2500.00
customer_age,int64,2500,52,18.00,69.00
customer_region,str,2500,5,NaN,NaN
customer_tenure_months,int64,2500,119,1.00,119.00
order_id,int64,2500,2500,50001.00,52500.00
order_value,float64,2500,2457,7.76,1983.81
items_quantity,int64,2500,6,1.00,6.00
discount_value,float64,2500,2050,0.02,230.33
payment_installments,int64,2500,11,1.00,11.00
delivery_time_days,int64,2500,13,2.00,14.00


In [ ]:
# Distribuição das variáveis numéricas, sem os identificadores
raw.drop(columns=["customer_id", "order_id"]).describe().T.round(2)


,count,mean,std,min,25%,50%,75%,max
customer_age,2500.0,43.40,14.89,18.00,31.00,43.00,56.00,69.00
customer_tenure_months,2500.0,61.32,34.48,1.00,31.00,62.00,91.00,119.00
order_value,2500.0,434.26,289.77,7.76,220.24,375.52,577.29,1983.81
items_quantity,2500.0,3.47,1.69,1.00,2.00,3.00,5.00,6.00
discount_value,2500.0,29.75,29.23,0.02,8.88,20.94,40.83,230.33
payment_installments,2500.0,6.00,3.16,1.00,3.00,6.00,9.00,11.00
delivery_time_days,2500.0,8.02,3.77,2.00,5.00,8.00,11.00,14.00
delivery_delay_days,2500.0,2.19,1.45,0.00,1.00,2.00,3.00,8.00
freight_value,2500.0,38.22,12.08,2.62,29.93,38.50,46.27,76.13
delivery_attempts,2500.0,2.01,0.82,1.00,1.00,2.00,3.00,3.00


**Observações**

- **Satisfação dos clientes:** Os indicadores `nps_score` e `csat_internal_score` apresentam médias e medianas relativamente baixas em suas escalas: 4,38 e 4,40 para o NPS, e 2,94 e 2,80 para o CSAT interno. Esses resultados sugerem uma possível insatisfação dos clientes, que deve ser investigada. A comparação entre os indicadores também pode ajudar a compreender se a percepção interna sobre a experiência do cliente acompanha as avaliações registradas.

- **Retenção de clientes:** A taxa de recompra em até 30 dias é de aproximadamente 9%. Embora esse resultado indique uma baixa proporção de recompras no período analisado, sua interpretação depende do contexto do negócio e do ciclo de compra esperado. A insatisfação dos clientes é uma possível hipótese explicativa, cuja relação com a recompra deverá ser investigada.

- **Logística e entregas:** O tempo de entrega apresenta média de 8,02 dias e desvio-padrão de 3,77 dias, indicando variação nos prazos observados. A avaliação desse desempenho depende dos prazos prometidos aos clientes e das referências do setor. O atraso médio é de 2,19 dias, com mediana de 2 dias. Além disso, o número médio de tentativas de entrega é de 2,01, o que justifica investigar a frequência de entregas que exigem tentativas adicionais. Em conjunto, essas características levantam a hipótese de possíveis ineficiências logísticas que podem comprometer a experiência do cliente. Essa hipótese deverá ser avaliada por meio da análise da relação entre os indicadores de entrega e os níveis de satisfação.

- **Atendimento ao cliente:** Os clientes apresentam, em média, 1,52 contatos com o atendimento, enquanto o tempo médio de resolução é de 5,49 dias. A avaliação desses indicadores depende dos padrões operacionais e dos níveis de serviço esperados para o negócio. A quantidade média de reclamações é de 4,15 por registro, com mediana de 4. É necessário compreender como essa variável foi definida e qual período de referência foi utilizado para avaliar se esse volume representa um problema operacional. A análise conjunta desses indicadores poderá ajudar a identificar possíveis gargalos no atendimento e investigar sua relação com a satisfação dos clientes.

- **Valores dos pedidos:** A variável `order_value` apresenta média de 434,26 e mediana de 375,52, com valor máximo de 1.983,81. A diferença entre a média e a mediana, associada à amplitude dos valores observados, sugere uma possível assimetria à direita e a presença de pedidos com valores elevados. Recomenda-se investigar a distribuição dos valores e verificar possíveis outliers, distinguindo observações legítimas de eventuais inconsistências nos dados.